# Rodent FOMO Detector — Google Colab Training

**Before running:**
1. Run `prepare_colab_dataset.py` locally → creates `datasets/colab_dataset/`
2. ZIP `colab_dataset/` and upload to your Google Drive
3. Runtime → Change runtime type → **GPU (T4)**
4. Run all cells in order

**Output:** `model_int8.tflite` — import into Edge Impulse as BYOM or embed directly.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q imagehash pillow tqdm
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Cell 2: Mount Drive and unzip dataset ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

# EDIT THIS: path to your uploaded ZIP on Drive
ZIP_PATH = '/content/drive/MyDrive/colab_dataset.zip'
DATASET_DIR = '/content/colab_dataset'

if not os.path.exists(DATASET_DIR):
    print('Unzipping dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Done.')
else:
    print('Dataset already unzipped.')

imgs = list(sorted(f for f in os.listdir(f'{DATASET_DIR}/images') if f.endswith('.jpg')))
print(f'Total images: {len(imgs)}')

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────
IMG_SIZE    = 96          # model input size
GRID_SIZE   = 12          # IMG_SIZE / 8 (FOMO stride)
NUM_CLASSES = 1           # rat only (background is class 0 implicitly)
ALPHA       = 0.35        # MobileNetV2 width multiplier
BATCH_SIZE  = 32
EPOCHS      = 80          # no time limit in Colab!
LR          = 1e-3
VAL_SPLIT   = 0.2
SEED        = 42

import numpy as np
import random
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Config OK')

In [ ]:
# ── Cell 4: Data loading helpers ──────────────────────────────────────────
from pathlib import Path
from PIL import Image

IMG_DIR = Path(DATASET_DIR) / 'images'
LBL_DIR = Path(DATASET_DIR) / 'labels'

def read_yolo(txt_path):
    """Returns list of [cls, cx, cy, w, h] normalised."""
    boxes = []
    p = Path(txt_path)
    if p.exists():
        for line in p.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                boxes.append([float(x) for x in parts])
    return boxes

def yolo_to_fomo(boxes, grid=GRID_SIZE, nc=NUM_CLASSES):
    """
    Convert YOLO box list to FOMO one-hot grid label.
    Shape: (grid, grid, nc+1)  where index 0 = background.
    """
    label = np.zeros((grid, grid), dtype=np.int32)  # 0 = background
    for b in boxes:
        cls_id = int(b[0])
        gx = min(int(b[1] * grid), grid - 1)
        gy = min(int(b[2] * grid), grid - 1)
        label[gy, gx] = cls_id + 1  # 1-indexed, 0 reserved for bg
    return np.eye(nc + 1, dtype=np.float32)[label]  # (grid, grid, nc+1)

def load_image(img_path):
    """Load image as float32 [0,1], RGB, resized to IMG_SIZE."""
    im = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.array(im, dtype=np.float32) / 255.0

# Collect all samples
all_names = sorted(p.stem for p in IMG_DIR.glob('*.jpg'))
print(f'Total samples: {len(all_names)}')

pos_names = [n for n in all_names if not n.startswith('bg_')]
bg_names  = [n for n in all_names if n.startswith('bg_')]
print(f'  Positives: {len(pos_names)}   Backgrounds: {len(bg_names)}')

In [ ]:
# ── Cell 5: Train / val split (scene-level) ───────────────────────────────
random.shuffle(pos_names)
random.shuffle(bg_names)

n_val_pos = int(len(pos_names) * VAL_SPLIT)
n_val_bg  = int(len(bg_names)  * VAL_SPLIT)

train_names = pos_names[n_val_pos:] + bg_names[n_val_bg:]
val_names   = pos_names[:n_val_pos] + bg_names[:n_val_bg]

random.shuffle(train_names)
random.shuffle(val_names)

print(f'Train: {len(train_names)}   Val: {len(val_names)}')

In [ ]:
# ── Cell 6: tf.data pipeline with augmentation ────────────────────────────

def make_dataset(names, augment=False, batch=BATCH_SIZE):
    """
    Build a tf.data.Dataset of (image, fomo_label) pairs.
    Augmentation: random flip, brightness, contrast, JPEG artefacts.
    """
    imgs, labels = [], []
    print(f'Loading {len(names)} samples...', end=' ')
    for name in names:
        img = load_image(IMG_DIR / f'{name}.jpg')
        boxes = read_yolo(LBL_DIR / f'{name}.txt')
        lbl = yolo_to_fomo(boxes)
        imgs.append(img)
        labels.append(lbl)
    imgs   = np.array(imgs,   dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)
    print('done.')

    ds = tf.data.Dataset.from_tensor_slices((imgs, labels))
    if augment:
        def aug_fn(img, lbl):
            # horizontal flip (mirror boxes too)
            if tf.random.uniform(()) > 0.5:
                img = tf.image.flip_left_right(img)
                lbl = tf.reverse(lbl, axis=[1])   # flip grid columns
            img = tf.image.random_brightness(img, 0.2)
            img = tf.image.random_contrast(img, 0.8, 1.2)
            img = tf.clip_by_value(img, 0.0, 1.0)
            # simulate low-res (ESP32 pixelation)
            def lowres(i):
                small = tf.image.resize(i, [32, 32], method='bilinear')
                return tf.image.resize(small, [IMG_SIZE, IMG_SIZE], method='nearest')
            img = tf.cond(tf.random.uniform(()) > 0.6, lambda: lowres(img), lambda: img)
            return img, lbl
        ds = ds.shuffle(2048, seed=SEED).map(aug_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_names, augment=True)
val_ds   = make_dataset(val_names,   augment=False)
print('Datasets ready.')

In [ ]:
# ── Cell 7: FOMO model ────────────────────────────────────────────────────

def build_fomo(num_classes=NUM_CLASSES, alpha=ALPHA):
    """
    FOMO: MobileNetV2 backbone truncated at stride-8 (12x12 feature map)
    + 1x1 conv classification head.
    Output: (batch, 12, 12, num_classes+1) with softmax.
    Compatible with Edge Impulse BYOM object detection.
    """
    # Preprocessing rescaling: [0,1] → [-1,1] for pretrained MobileNetV2
    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input')
    x   = tf.keras.layers.Rescaling(scale=2.0, offset=-1.0)(inp)

    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        alpha=alpha,
        include_top=False,
        weights='imagenet'
    )

    # Find the 12x12 feature layer (stride 8)
    feature_out = None
    for layer in backbone.layers:
        try:
            s = layer.output.shape
            if len(s) == 4 and s[1] == GRID_SIZE and s[2] == GRID_SIZE:
                feature_out = layer.output
                print(f'Using feature layer: {layer.name}  shape: {s}')
        except Exception:
            pass

    assert feature_out is not None, 'Could not find 12x12 feature layer!'

    # Build truncated backbone
    trunc = tf.keras.Model(inputs=backbone.input, outputs=feature_out, name='backbone')

    # Chain: rescaled input → backbone → FOMO head
    feats = trunc(x)
    n_out = num_classes + 1  # 0=bg, 1=rat
    out   = tf.keras.layers.Conv2D(
        n_out, kernel_size=1, padding='same',
        activation='softmax', name='fomo_head'
    )(feats)

    model = tf.keras.Model(inputs=inp, outputs=out, name='fomo')
    return model

model = build_fomo()
model.summary(line_length=80)

In [ ]:
# ── Cell 8: Loss, metrics and compile ─────────────────────────────────────

def fomo_loss(y_true, y_pred, bg_weight=0.1):
    """
    Weighted categorical cross-entropy per grid cell.
    Background cells get lower weight to handle the severe class imbalance
    (12*12=144 cells per image, only a few contain rats).
    """
    is_bg   = y_true[..., 0]                              # (batch,12,12)
    weight  = tf.where(is_bg > 0.5, bg_weight, 1.0)      # bg vs object
    cce     = -tf.reduce_sum(y_true * tf.math.log(tf.clip_by_value(y_pred, 1e-7, 1.0)), axis=-1)
    return tf.reduce_mean(cce * weight)


def fomo_f1(y_true, y_pred, threshold=0.5):
    """Approximate F1 on object cells only (ignores background cells)."""
    # Ground truth: any cell with class > 0
    true_obj  = tf.reduce_any(y_true[..., 1:] > 0.5, axis=-1)          # (B,12,12) bool
    pred_obj  = tf.reduce_any(y_pred[..., 1:] > threshold, axis=-1)
    tp = tf.reduce_sum(tf.cast(true_obj & pred_obj, tf.float32))
    fp = tf.reduce_sum(tf.cast(~true_obj & pred_obj, tf.float32))
    fn = tf.reduce_sum(tf.cast(true_obj & ~pred_obj, tf.float32))
    precision = tp / (tp + fp + 1e-7)
    recall    = tp / (tp + fn + 1e-7)
    return 2 * precision * recall / (precision + recall + 1e-7)


model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss=fomo_loss,
    metrics=[fomo_f1]
)
print('Model compiled.')

In [ ]:
# ── Cell 9: Training ──────────────────────────────────────────────────────
import os

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'best_fomo.keras', monitor='val_fomo_f1',
        mode='max', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=8, min_lr=1e-5, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_fomo_f1', mode='max',
        patience=15, restore_best_weights=True, verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ── Cell 10: Plot training curves ─────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'],     label='train loss')
ax1.plot(history.history['val_loss'], label='val loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)

ax2.plot(history.history['fomo_f1'],     label='train F1')
ax2.plot(history.history['val_fomo_f1'], label='val F1')
ax2.set_title('FOMO F1'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()
print('Best val F1:', max(history.history['val_fomo_f1']))

In [ ]:
# ── Cell 11: Export TFLite int8 ───────────────────────────────────────────

# Representative dataset for calibration (100 random val images)
val_imgs = np.array([load_image(IMG_DIR / f'{n}.jpg') for n in val_names[:100]])

def representative_dataset():
    for i in range(len(val_imgs)):
        yield [val_imgs[i:i+1].astype(np.float32)]

# Convert to TFLite float32 first (sanity check)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float = converter.convert()
print(f'Float32 TFLite size: {len(tflite_float)/1024:.1f} KB')

# Convert to int8
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations              = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset     = representative_dataset
converter.target_spec.supported_ops  = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type       = tf.int8
converter.inference_output_type      = tf.int8

tflite_int8 = converter.convert()
with open('model_int8.tflite', 'wb') as f:
    f.write(tflite_int8)
print(f'Int8 TFLite size: {len(tflite_int8)/1024:.1f} KB  → model_int8.tflite')

In [ ]:
# ── Cell 12: Verify int8 model on a few val samples ──────────────────────

interp = tf.lite.Interpreter('model_int8.tflite')
interp.allocate_tensors()
inp_det  = interp.get_input_details()[0]
out_det  = interp.get_output_details()[0]

print('Input  :', inp_det['dtype'], inp_det['shape'],
      'scale=', inp_det['quantization'][0], 'zero=', inp_det['quantization'][1])
print('Output :', out_det['dtype'], out_det['shape'],
      'scale=', out_det['quantization'][0], 'zero=', out_det['quantization'][1])

# Run inference on 5 val samples
in_scale, in_zero = inp_det['quantization']
out_scale, out_zero = out_det['quantization']

for name in val_names[:5]:
    img_f = load_image(IMG_DIR / f'{name}.jpg')
    img_q = np.round(img_f / in_scale + in_zero).clip(-128, 127).astype(np.int8)
    interp.set_tensor(inp_det['index'], img_q[np.newaxis])
    interp.invoke()
    out_q  = interp.get_tensor(out_det['index'])    # (1,12,12,2) int8
    out_f  = (out_q.astype(np.float32) - out_zero) * out_scale   # dequantize
    rat_conf = out_f[0, :, :, 1].max()             # max rat confidence in grid
    has_rat  = not name.startswith('bg_')
    print(f'  {name}  has_rat={has_rat}  max_rat_conf={rat_conf:.3f}')

In [ ]:
# ── Cell 13: Save to Google Drive ─────────────────────────────────────────
import shutil

DRIVE_OUT = '/content/drive/MyDrive/rodent_fomo_output'
os.makedirs(DRIVE_OUT, exist_ok=True)

shutil.copy('model_int8.tflite',   f'{DRIVE_OUT}/model_int8.tflite')
shutil.copy('best_fomo.keras',     f'{DRIVE_OUT}/best_fomo.keras')
shutil.copy('training_curves.png', f'{DRIVE_OUT}/training_curves.png')
print('Saved to Drive:', DRIVE_OUT)

## Next steps — deploy to ESP32-CAM

### Option A — Import into Edge Impulse (recommended, no code change)
1. Open your EI project → **Create Impulse**.
2. Add **Image** block (96×96, Grayscale).
3. Add **Object Detection (FOMO)** learning block → choose **"Bring Your Own Model"**.
4. Upload `model_int8.tflite`.
5. Set labels: `background, rat`.
6. **Deployment → Arduino library → Build** → download zip → install in Arduino IDE → reflash.

### Option B — Use TFLite Micro directly (advanced)
Replace EI inference code with TFLite Micro in the firmware.
Requires rewriting `rodent_webserver.ino` to call `tflite::MicroInterpreter` directly.

---
**Model output format (for custom inference):**
- Shape: `(1, 12, 12, 2)` int8
- Channel 0 = background probability, Channel 1 = rat probability
- Each cell covers an 8×8 pixel region of the 96×96 input
- Detection threshold: 0.5–0.7 on dequantized rat channel